# Linear Regression

This notebook shows the linear regression workflow using the package implementation. The formulas are shown in small cells, while training and evaluation use the reusable project code.

Author: Pasquale Marzaioli

## Model Formula

Linear regression predicts a numeric target with:

```text
y_hat = X @ w + b
```

The loss used here is mean squared error:

```text
loss = mean((y_hat - y) ** 2)
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from ml_from_scratch import LinearRegressionGD, mean_squared_error
from ml_from_scratch.datasets import make_regression_data
from ml_from_scratch.plotting import plot_loss_curve, plot_regression_fit
from ml_from_scratch.preprocessing import (
    normalize_features,
    train_test_split,
    transform_features,
)

## A Tiny Formula Check

This cell computes one prediction and one loss directly, just to connect the formula to NumPy.

In [ ]:
X_demo = np.array([[2.0]])
weights_demo = np.array([3.0])
bias_demo = 2.0
y_demo = np.array([8.5])

prediction_demo = X_demo @ weights_demo + bias_demo
loss_demo = mean_squared_error(y_demo, prediction_demo)

print(f"prediction: {prediction_demo[0]:.3f}")
print(f"MSE: {loss_demo:.3f}")

## Generate And Split Data

The dataset helper creates data from a known line plus Gaussian noise. The split keeps test data separate from training.

In [ ]:
X, y = make_regression_data(
    n_samples=80,
    weights=np.array([3.0]),
    bias=2.0,
    noise=1.0,
    random_state=7,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=7,
)

print(X_train.shape, X_test.shape)
print(f"first training row: x={X_train[0, 0]:.3f}, y={y_train[0]:.3f}")

## Normalize Features

Normalization puts features on a similar scale, which makes gradient descent easier to tune. The test set uses the training means and scales.

In [ ]:
X_train_normalized, means, scales = normalize_features(X_train)
X_test_normalized = transform_features(X_test, means, scales)

print(f"training mean after normalization: {np.mean(X_train_normalized):.3f}")
print(f"training std after normalization: {np.std(X_train_normalized):.3f}")

## Train The Model

`LinearRegressionGD` stores the loss before each gradient descent update, so we can inspect convergence.

In [ ]:
model = LinearRegressionGD(learning_rate=0.01, n_iterations=1000)
model.fit(X_train_normalized, y_train)

raw_weight = model.weights_[0] / scales[0]
raw_bias = model.bias_ - float(np.sum(model.weights_ * means / scales))

print(f"raw-scale weight: {raw_weight:.3f}")
print(f"raw-scale bias: {raw_bias:.3f}")
print(f"first loss: {model.loss_history_[0]:.3f}")
print(f"final loss: {model.loss_history_[-1]:.3f}")

In [ ]:
ax = plot_loss_curve(model.loss_history_)
plt.show()

In [ ]:
ax = plot_regression_fit(
    X_train,
    y_train,
    model,
    prediction_X=X_train_normalized,
)
plt.show()

## Evaluate On Test Data

Test error estimates how well the learned line behaves on data that was not used for gradient updates.

In [ ]:
test_predictions = model.predict(X_test_normalized)
test_mse = mean_squared_error(y_test, test_predictions)

print(f"test MSE: {test_mse:.3f}")